In [ ]:
import cv2
import os
import shutil
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
from ultralytics import YOLO
import threading

# ==========================================
# 1. 모델 경로 설정
# ==========================================
MODEL_PATH = "/Users/hisj0hn/Documents/for_code/miniproject/learning/yolov8n.pt"

class AutoLabelerApp:
    def __init__(self):
        self.root = tk.Tk()
        self.root.title("YOLO 통합 자동 라벨러 (안정화 버전)")
        self.root.geometry("450x250")
        
        # UI 요소
        self.status_label = tk.Label(self.root, text="영상 폴더를 선택해주세요.", pady=10)
        self.status_label.pack()

        self.progress_var = tk.DoubleVar()
        self.progress_bar = ttk.Progressbar(self.root, variable=self.progress_var, maximum=100, length=350)
        self.progress_bar.pack(pady=10)

        self.detail_label = tk.Label(self.root, text="", fg="blue", wraplength=400)
        self.detail_label.pack()

        self.start_button = tk.Button(self.root, text="폴더 선택 및 시작", command=self.start_thread)
        self.start_button.pack(pady=10)

    def start_thread(self):
        thread = threading.Thread(target=self.run_pipeline)
        thread.daemon = True
        thread.start()

    def update_status_safe(self, main_txt, detail_txt, progress):
        """스레드 세이프한 UI 업데이트"""
        self.root.after(0, lambda: self._update_ui(main_txt, detail_txt, progress))

    def _update_ui(self, main_txt, detail_txt, progress):
        self.status_label.config(text=main_txt)
        self.detail_label.config(text=detail_txt)
        self.progress_var.set(progress)

    def run_pipeline(self):
        target_dir = filedialog.askdirectory(title="영상 파일 폴더 선택")
        if not target_dir:
            return

        self.start_button.config(state="disabled")
        
        try:
            # 2. 구조 생성
            dataset_root = os.path.join(target_dir, "captured")
            images_folder = os.path.join(dataset_root, "images")
            labels_folder = os.path.join(dataset_root, "labels")
            os.makedirs(images_folder, exist_ok=True)
            os.makedirs(labels_folder, exist_ok=True)

            valid_extensions = ('.mp4', '.avi', '.mov', '.mkv', '.wmv')
            video_files = [f for f in os.listdir(target_dir) if f.lower().endswith(valid_extensions)]
            
            if not video_files:
                messagebox.showerror("오류", "영상 파일이 없습니다.")
                self.start_button.config(state="normal")
                return

            # STEP 1: 프레임 추출
            total_videos = len(video_files)
            captured_images = []
            
            for i, video_filename in enumerate(video_files):
                video_path = os.path.join(target_dir, video_filename)
                video_name = os.path.splitext(video_filename)[0]
                
                progress = (i / total_videos) * 40
                self.update_status_safe("단계 1: 프레임 캡처 중...", f"[{i+1}/{total_videos}] {video_filename}", progress)

                cap = cv2.VideoCapture(video_path)
                fps = cap.get(cv2.CAP_PROP_FPS)
                interval = max(int(fps), 1) if fps > 0 else 30
                
                count = 0
                video_saved_count = 0
                while True:
                    ret, frame = cap.read()
                    if not ret: break
                    if count % interval == 0:
                        file_name = f"{video_name}_{video_saved_count:04d}.jpg"
                        save_path = os.path.join(images_folder, file_name)
                        # 메모리 효율을 위해 imwrite 성공 시에만 리스트에 추가
                        if cv2.imwrite(save_path, frame):
                            captured_images.append(save_path)
                            video_saved_count += 1
                    count += 1
                cap.release()

            # STEP 2: YOLO 자동 라벨링 (Stream 방식)
            self.update_status_safe("단계 2: YOLO 분석 중...", "모델 로딩 중...", 45)
            model = YOLO(MODEL_PATH)
            
            total_imgs = len(captured_images)
            label_count = 0

            # stream=True를 사용하여 메모리 부하 방지
            # results는 generator로 반환됨
            results = model.predict(
                source=images_folder,
                save_txt=True,
                project=dataset_root,
                name="temp_process",
                exist_ok=True,
                stream=True,  # 핵심: 대량의 파일을 처리할 때 필수
                conf=0.25     # 기본 신뢰도 설정
            )

            for idx, r in enumerate(results):
                if idx % 5 == 0: # 5개마다 UI 업데이트 (성능 저하 방지)
                    prog = 50 + (idx / total_imgs) * 45
                    self.update_status_safe("단계 2: YOLO 분석 중...", f"분석 중: {idx}/{total_imgs}", prog)
            
            # 결과 정리 (shutil.move의 안정성을 위해 이동 전 잠시 대기 혹은 체크)
            temp_labels_dir = os.path.join(dataset_root, "temp_process", "labels")
            if os.path.exists(temp_labels_dir):
                label_files = [f for f in os.listdir(temp_labels_dir) if f.endswith(".txt")]
                for f in label_files:
                    src = os.path.join(temp_labels_dir, f)
                    dst = os.path.join(labels_folder, f)
                    if os.path.exists(dst): os.remove(dst) # 중복 제거
                    shutil.move(src, dst)
                    label_count += 1

            # 임시 폴더 삭제 시도 (에러 방지를 위해 ignore_errors=True)
            shutil.rmtree(os.path.join(dataset_root, "temp_process"), ignore_errors=True)

            self.update_status_safe("완료!", f"이미지: {total_imgs}장 / 라벨: {label_count}개", 100)
            messagebox.showinfo("성공", f"작업 완료!\n\n경로: {dataset_root}")

        except Exception as e:
            messagebox.showerror("오류 발생", f"상세 내용: {str(e)}")
        
        finally:
            self.start_button.config(state="normal")

    def run(self):
        self.root.mainloop()

if __name__ == "__main__":
    app = AutoLabelerApp()
    app.run()